In [ ]:
# Assignment 8 — NLP using Python II
# Feature extraction, sentiment analysis, similarity, and text generation.

In [ ]:
import re, string
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from collections import Counter
for resource in ['punkt','punkt_tab','stopwords','wordnet','omw-1.4']:
    nltk.download(resource, quiet=True)
stop_words=set(stopwords.words('english'))

In [ ]:
# Q1. Text preprocessing and frequency

In [ ]:
paragraph = ("Artificial intelligence fascinates me because it helps people solve complex problems. "
             "Machine learning can discover patterns in data and support better decisions. "
             "I enjoy learning Python, building small projects, and exploring useful technology. "
             "Responsible AI should be transparent, fair, and helpful to society.")
lower_clean = re.sub(r'[^a-zA-Z\\s]', '', paragraph.lower())
words = word_tokenize(lower_clean)
sentences = sent_tokenize(paragraph)
split_words = paragraph.split()
nltk_words = word_tokenize(paragraph)
filtered = [w for w in words if w not in stop_words]
print("Lowercase and punctuation removed:", lower_clean)
print("Sentences:", sentences)
print("split():", split_words)
print("word_tokenize():", nltk_words)
print("Filtered tokens:", filtered)
print("Frequency:", Counter(filtered))

In [ ]:
# Comparison: split() primarily splits on whitespace. word_tokenize() uses language-aware tokenization rules and generally separates punctuation and handles contractions more deliberately.

In [ ]:
# Q2. Regex, stemming and lemmatization

In [ ]:
alpha_words = re.findall(r'\\b[A-Za-z]+\\b', paragraph.lower())
alpha_no_stop = [w for w in alpha_words if w not in stop_words]
stemmer = PorterStemmer()
stemmed = [stemmer.stem(w) for w in alpha_no_stop]
from nltk.stem import WordNetLemmatizer
lemma = WordNetLemmatizer()
lemmatized = [lemma.lemmatize(w) for w in alpha_no_stop]
print("Alphabetic words:", alpha_words)
print("Without stopwords:", alpha_no_stop)
print("Stemmed:", stemmed)
print("Lemmatized:", lemmatized)

In [ ]:
# Stemming vs. lemmatization: stemming mechanically trims affixes and can produce fragments. Lemmatization consults lexical forms and can produce actual words. Stemming is useful for fast normalization; lemmatization is preferable when readability and meaning matter.

In [ ]:
# Q3. Bag of Words and TF-IDF

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
texts = [
    "AI assistants can help students study and organize information.",
    "Data analytics uses statistics to find patterns and support decisions.",
    "Python programming builds useful applications and machine learning models."
]
count_vec = CountVectorizer()
bow = count_vec.fit_transform(texts)
print("Vocabulary:", count_vec.get_feature_names_out())
print("Bag of Words matrix:\\n", bow.toarray())

tfidf_vec = TfidfVectorizer()
tfidf = tfidf_vec.fit_transform(texts)
terms = tfidf_vec.get_feature_names_out()
for i, row in enumerate(tfidf.toarray()):
    top_idx = row.argsort()[-3:][::-1]
    print(f"Text {i+1} top keywords:", [(terms[j], round(row[j],3)) for j in top_idx])

In [ ]:
# Q4. Similarity of two technology descriptions

In [ ]:
text_ai = ("Artificial intelligence learns patterns from data. AI systems automate tasks and make predictions. "
            "Machine learning is an important part of artificial intelligence.")
text_blockchain = ("Blockchain stores records in linked blocks. Distributed ledgers support transparent transactions. "
                   "Cryptography protects blockchain data.")
def preprocess_for_similarity(text):
    tokens = re.findall(r'\\b[a-z]+\\b', text.lower())
    return set(t for t in tokens if t not in stop_words)
set_a, set_b = preprocess_for_similarity(text_ai), preprocess_for_similarity(text_blockchain)
jaccard = len(set_a & set_b)/len(set_a | set_b) if (set_a | set_b) else 0
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = TfidfVectorizer().fit_transform([text_ai,text_blockchain])
cosine = cosine_similarity(similarity_matrix)[0,1]
print("Jaccard:", jaccard)
print("TF-IDF cosine:", cosine)

In [ ]:
# Interpretation: Jaccard compares overlap between distinct word sets and treats every word equally. TF-IDF cosine considers vector weights and can be more informative when words vary in importance, though both are limited by vocabulary and do not understand meaning or synonyms.

In [ ]:
# Q5. Sentiment analysis and word cloud

In [ ]:
reviews = [
 "This product is excellent and easy to use. I love the quality.",
 "Terrible experience. The product broke and support was unhelpful.",
 "The product is okay and arrived on time.",
 "Fantastic value, fast delivery, and wonderful customer service!",
 "I dislike this product; it is disappointing and unreliable."
]
# VADER
nltk.download('vader_lexicon', quiet=True)
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()
results=[]
positive_reviews=[]
for review in reviews:
    scores=sia.polarity_scores(review)
    compound=scores['compound']
    label='Positive' if compound>=0.05 else ('Negative' if compound<=-0.05 else 'Neutral')
    results.append({'review':review,'polarity/compound':compound,'label':label,'scores':scores})
    if label=='Positive': positive_reviews.append(review)
display(pd.DataFrame(results))

In [ ]:
# Word cloud from positive reviews; may require the wordcloud package.
# Install if needed: %pip install wordcloud
from wordcloud import WordCloud
positive_text=' '.join(positive_reviews)
if positive_text.strip():
    wc=WordCloud(width=900,height=400,background_color='white').generate(positive_text)
    import matplotlib.pyplot as plt
    plt.figure(figsize=(12,5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.show()
else:
    print("No positive reviews to plot.")

In [ ]:
# Q6. Simple LSTM text generation

In [ ]:
# Install TensorFlow if needed: %pip install tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import numpy as np

training_text = ("Technology can improve learning when people use it thoughtfully. "
 "Students explore ideas, ask questions, practice skills, and create projects. "
 "Machine learning models find patterns and make predictions from examples. "
 "Responsible innovation values privacy, fairness, creativity, and collaboration. "
 "Small experiments help learners understand complex concepts and develop confidence. "
 "Curiosity encourages people to keep learning and build useful tools.")
tokenizer=Tokenizer()
tokenizer.fit_on_texts([training_text])
sequences=tokenizer.texts_to_sequences([training_text])[0]
vocab_size=len(tokenizer.word_index)+1
input_sequences=[]
for i in range(2,len(sequences)+1):
    input_sequences.append(sequences[:i])
max_len=max(map(len,input_sequences))
input_sequences=pad_sequences(input_sequences,maxlen=max_len,padding='pre')
X_train=input_sequences[:,:-1]
y_train=input_sequences[:,-1]
model=Sequential([
    Embedding(vocab_size,32,input_length=X_train.shape[1]),
    LSTM(64),
    Dense(vocab_size,activation='softmax')
])
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
model.fit(X_train,y_train,epochs=80,verbose=1)

In [ ]:
def generate_text(seed, n_words=20):
    result=seed
    for _ in range(n_words):
        seq=tokenizer.texts_to_sequences([result])[0]
        seq=pad_sequences([seq],maxlen=X_train.shape[1],padding='pre')
        probs=model.predict(seq,verbose=0)[0]
        next_id=int(np.argmax(probs))
        next_word=tokenizer.index_word.get(next_id,'')
        if not next_word: break
        result += ' ' + next_word
    return result

print(generate_text('technology', 20))
print(generate_text('students', 20))
print(generate_text('learning', 20))

In [ ]:
# Note: This tiny training corpus is for demonstration. A small LSTM trained on a short paragraph can overfit and produce repetitive or incoherent output; meaningful generation needs substantially more text.